# Load input data

In [1]:
from functions import get_experiment_data

X_df1, meta_df1, batch_df1, _ = get_experiment_data(
    design_id="df1",
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df1
Design name             : full_raw_norm
Description             : Full raw HIVRC, normalized, standalone LatentGEE
Aggregation             : None
Normalize               : True
Cleanset Filtering      : False
Subset studies          : None
OTU zero-prev           : 0.01
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (1032, 2485)
meta_data               : (1032, 11)
n_batches               : 17


# Trial investigation

In [4]:
import pandas as pd
import glob

# df1 CSV 파일 찾기
# CSV 파일 로드 및 병합
files = [
    "/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_2026-05-13.csv"
]

df_list = [pd.read_csv(f) for f in files]
df1_p1_res = pd.concat(df_list, ignore_index=True)
print(f"전체 trial 수: {len(df1_p1_res)}")

# 필터링
df_clean = df1_p1_res[
    (df1_p1_res['permanova'] > 0.01) &
    (df1_p1_res['permanova'] < 0.1) &
    # (df1_p1_res['noise_ratio'] < 0.65) &
    (df1_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

전체 trial 수: 960
의미있는 trial 수: 1
     trial_number  permanova  n_clusters  noise_ratio  cutoff
219           701   0.068199          11      0.70155    0.07


In [12]:
import optuna

study = optuna.load_study(
    study_name="latentgee_optuna_df1",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
)

# 상태별 trial 수 확인
from collections import Counter
state_counts = Counter(str(t.state) for t in study.trials)
for state, count in state_counts.items():
    print(f"{state}: {count}")

TrialState.COMPLETE: 2000


In [ ]:
# n_clusters 기준 완화
for min_k in [10, 8, 5, 3]:
    filtered = df1_p1_res[
        (df1_p1_res['permanova'] > 0.01) &
        (df1_p1_res['permanova'] < 0.1) &
        (df1_p1_res['noise_ratio'] < 0.65) &
        (df1_p1_res['n_clusters'] >= min_k)
    ]
    print(f"n_clusters >={min_k}: {len(filtered)}개")


# 조건 완화한 best trial 확인
df_clean2 = df1_p1_res[
        (df1_p1_res['permanova'] > 0.01) &
        (df1_p1_res['permanova'] < 0.1) &
        (df1_p1_res['noise_ratio'] < 0.65)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"\n완화된 필터 결과: {len(df_clean2)}개")
print(df_clean2[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

n_clusters >=10: 0개
n_clusters >=8: 0개
n_clusters >=5: 0개
n_clusters >=3: 13개

완화된 필터 결과: 20개
     trial_number  permanova  n_clusters  noise_ratio  cutoff
91            216   0.057345           2     0.007752    0.10
502          1310   0.035267           2     0.011628    0.10
543          1361   0.038773           2     0.011628    0.10
252           847   0.046363           2     0.018411    0.10
249           844   0.065336           2     0.022287    0.10
250           843   0.042546           2     0.023256    0.10
599          1427   0.031917           2     0.024225    0.10
925          1931   0.055875           2     0.026163    0.10
304           979   0.030515           2     0.027132    0.10
878          1846   0.026605           2     0.029070    0.10
912          1909   0.028406           2     0.030039    0.10
488          1293   0.027966           2     0.031977    0.10
137           328   0.050081           2     0.032946    0.10
539          1357   0.030000          

In [13]:
# noise_ratio 필터 없이 전체 분포 확인
print("noise_ratio 분포:")
print(df1_p1_res['noise_ratio'].describe())

print("\npermanova 0.01~0.1 구간 trial 수:")
print(len(df1_p1_res[
    (df1_p1_res['permanova'] > 0.01) &
    (df1_p1_res['permanova'] < 0.1)
]))

print("\nnoise_ratio < 0.65인 trial 수:")
print(len(df1_p1_res[df1_p1_res['noise_ratio'] < 0.65]))

noise_ratio 분포:
count    960.000000
mean       0.487520
std        0.330333
min        0.007752
25%        0.161822
50%        0.436047
75%        0.846172
max        0.949612
Name: noise_ratio, dtype: float64

permanova 0.01~0.1 구간 trial 수:
673

noise_ratio < 0.65인 trial 수:
568


# Best trial selection (trial 701)

trial 216 — noise가 거의 없고 안정적이지만 클러스터 2개라 GEE가 단순해요.
trial 701 — noise가 70%로 높지만 클러스터 11개라 17개 study의 batch 구조를 더 잘 반영할 수 있어요.
Phase 2 결과를 보고 PERMANOVA R², kBET 등 최종 metrics로 비교해서 더 나은 쪽을 선택

In [14]:
print(df1_p1_res[df1_p1_res['trial_number'] == 216].T)

                              91
cutoff                       0.1
trial_number                 216
base_dim                     768
n_layers                       4
latent_dim                    26
activation                  relu
strategy                constant
dropout_rate                 0.3
epochs                       100
learning_rate           0.000047
loss                    1.472543
permanova               0.057345
n_clusters                     2
noise_ratio             0.007752
min_cluster_size               4
min_samples_token            5.0
batch_size                   128
init               xavier_normal
beta_kl                 0.057969
weight_decay            0.000098
grad_clip_norm          0.607809
csm                          eom
kl_warmup_ratio         0.200919
norm                   batchnorm
scheduler                 cosine


In [6]:
print(df1_p1_res[df1_p1_res['trial_number'] == 701].T)

                              219
cutoff                       0.07
trial_number                  701
base_dim                      512
n_layers                        4
latent_dim                     54
activation                   relu
strategy                   double
dropout_rate                  0.3
epochs                         80
learning_rate            0.000165
loss                      0.73285
permanova                0.068199
n_clusters                     11
noise_ratio               0.70155
min_cluster_size                5
min_samples_token             5.0
batch_size                     32
init               xavier_uniform
beta_kl                  0.023279
weight_decay             0.000289
grad_clip_norm           0.695313
csm                          leaf
kl_warmup_ratio          0.327801
norm                    batchnorm
scheduler                  cosine


# Run phase 2

In [15]:
from prototype_updated_phase2 import main
# trial 216
main(
    experiment_name="df1",
    phase=2,
    best_trial_number=216,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_2026-05-13.csv"
)

# trial 701
main(
    experiment_name="df1",
    phase=2,
    best_trial_number=701,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_2026-05-13.csv"
)

2026-05-18 10:31:56 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/latentgee_prototype_cutoff_df1_pid1714450_2026-05-18.log
2026-05-18 10:31:56 | LatentGEE 시작 | experiment=df1 | phase=2
2026-05-18 10:31:56 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/config_used.yaml
2026-05-18 10:31:56 | python == 3.10.20
2026-05-18 10:31:56 | torch == 2.2.2
2026-05-18 10:31:56 | numpy == 1.26.4
2026-05-18 10:31:56 | scikit-learn == 1.7.2
2026-05-18 10:31:56 | optuna == 3.6.1
2026-05-18 10:31:56 | hdbscan == 0.8.41
Design ID               : df1
Design name             : full_raw_norm
Description             : Full raw HIVRC, normalized, standalone LatentGEE
Aggregation             : None
Normalize               : True
Cleanset Filtering      : False
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table      

In [17]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.1
X_df1_216_filtered, meta_df1_216_filtered, batch_df1_216_filtered = zero_filter(X_df1, meta_df1, best_cutoff)
X_df1_216_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/X_df1_filtered_cutoff{best_cutoff}.csv", index=True)
meta_df1_216_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/meta_df1_filtered_cutoff{best_cutoff}.csv", index=True)
batch_df1_216_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/batch_df1_filtered_cutoff{best_cutoff}.csv", index=True)


best_cutoff = 0.07
X_df1_701_filtered, meta_df1_701_filtered, batch_df1_701_filtered = zero_filter(X_df1, meta_df1, best_cutoff)
X_df1_701_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/X_df1_filtered_cutoff{best_cutoff}.csv", index=True)
meta_df1_701_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/meta_df1_filtered_cutoff{best_cutoff}.csv", index=True)
batch_df1_701_filtered.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/batch_df1_filtered_cutoff{best_cutoff}.csv", index=True)




In [23]:
import numpy as np
from functions import evaluate_batch_correction

# X_corrected 로드
trial=216
cutoff=0.1
X_corrected_df1_216 = pd.read_csv(
    f"/DATA/WGS_study/YSL/projects/latentgee/examples/results/df1/phase2/df1_X_corrected_trial{trial}_cutoff{cutoff}_2026-05-18.csv",
    index_col=0
)

# inf, NaN 처리
X_corrected_df1_216_clean = X_corrected_df1_216.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df1_216_clean.sum(axis=1).replace(0, 1)
X_corrected_df1_216_clean = X_corrected_df1_216_clean.div(row_sums, axis=0)

print(f"=============== trial: {trial}, cutoff: {cutoff}")
print(f"shape: {X_corrected_df1_216_clean.shape}")
print(f"NaN 수: {X_corrected_df1_216_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df1_216_clean.values).sum()}")
print()

df1_216_result = evaluate_batch_correction(
    X_before     = X_df1_216_filtered,
    X_after      = X_corrected_df1_216_clean,
    batch_labels = batch_df1_216_filtered,
    bio_labels   = meta_df1_216_filtered['hivstatus'],
    renormalize  = True,
    label        = "df1 — LatentGEE"
)


=============== trial: 216, cutoff: 0.1
shape: (1032, 468)
NaN 수: 0
inf 수: 0


  df1 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2174  0.0184 -0.1990
PERMANOVA R² (bio) ↑    0.0027  0.0003 -0.0024
kBET acceptance rate ↑  0.0785  0.8295  0.7510
ASW (batch) → 0        -0.0478 -0.1049 -0.0571
ASW (bio) ↑            -0.0229 -0.0004  0.0226



In [24]:
import numpy as np
from functions import evaluate_batch_correction

# X_corrected 로드
trial=701
cutoff=0.07
X_corrected_df1_701 = pd.read_csv(
    f"/DATA/WGS_study/YSL/projects/latentgee/examples/results/df1/phase2/df1_X_corrected_trial{trial}_cutoff{cutoff}_2026-05-18.csv",
    index_col=0
)

# inf, NaN 처리
X_corrected_df1_701_clean = X_corrected_df1_701.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df1_701_clean.sum(axis=1).replace(0, 1)
X_corrected_df1_701_clean = X_corrected_df1_701_clean.div(row_sums, axis=0)

print(f"=============== trial: {trial}, cutoff: {cutoff}")
print(f"shape: {X_corrected_df1_701_clean.shape}")
print(f"NaN 수: {X_corrected_df1_701_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df1_701_clean.values).sum()}")
print()

df1_216_result = evaluate_batch_correction(
    X_before     = X_df1_701_filtered,
    X_after      = X_corrected_df1_701_clean,
    batch_labels = batch_df1_701_filtered,
    bio_labels   = meta_df1_701_filtered['hivstatus'],
    renormalize  = True,
    label        = "df1 — LatentGEE"
)


=============== trial: 701, cutoff: 0.07
shape: (1032, 618)
NaN 수: 0
inf 수: 0


  df1 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2146  0.2381  0.0235
PERMANOVA R² (bio) ↑    0.0027  0.0015 -0.0012
kBET acceptance rate ↑  0.0853  0.3818  0.2965
ASW (batch) → 0        -0.0435 -0.1199 -0.0763
ASW (bio) ↑            -0.0229  0.0053  0.0281



결론 - trial 216이 압도적으로 낫다.
trial 701은 오히려 before보다 나빠졌다.
- PERMANOVA R²(batch): 0.2146 → 0.2381 (증가 = 나빠짐)
- kBET: 0.0853 → 0.3818 (개선됐지만 trial 216에 비해 훨씬 낮음)

--> noise_ratio 70%의 영향이 그대로 드러난 것.
샘플 70%가 노이즈로 처리되다 보니 GEE residualization이 제대로 작동하지 않은 것.

Trial 216이 df1의 best trial이다.

PERMANOVA R²(batch) 91.5% 감소 (0.2174 → 0.0184)
kBET 0.0785 → 0.8295로 대폭 향상
n_clusters=2라는 한계에도 불구하고 강력한 보정 성능